In [36]:
import pandas as pd
import numpy as np

# fetch the data we will be using to train and validate our model
training_data = pd.read_csv('data/train.csv')
TIME_HORIZONS = [12, 24, 48, 72]
# print the dimensions of the training data to understand its size
print(training_data.shape)
training_data.head(1)

(221, 37)


,event_id,num_perimeters_0_5h,dt_first_last_0_5h,low_temporal_resolution_0_5h,area_first_ha,area_growth_abs_0_5h,area_growth_rel_0_5h,area_growth_rate_ha_per_h,log1p_area_first,log1p_growth,...,dist_fit_r2_0_5h,alignment_cos,alignment_abs,cross_track_component,along_track_speed,event_start_hour,event_start_dayofweek,event_start_month,time_to_hit_hours,event
0,10892457,3,4.265188,0,79.696304,2.875935,0.036086,0.674281,4.390693,1.354787,...,0.886373,-0.054649,0.054649,-1.937219,-0.106026,19,4,5,18.892512,0


In [37]:
from sklearn.model_selection import train_test_split
# split the data into training and validation sets. set aside 20% of the data for validation
train, val = train_test_split(training_data, test_size=0.2, random_state=42)

In [38]:
TARGETS = ['event', 'time_to_hit_hours']
features = [
    'area_first_ha',
    'area_growth_abs_0_5h',
    'area_growth_rel_0_5h',
    'centroid_displacement_m',
    'centroid_speed_m_per_h',
    'spread_bearing_sin',
    'spread_bearing_cos',
    'num_perimeters_0_5h',
    'dt_first_last_0_5h',
    'low_temporal_resolution_0_5h'
]
train_iteration_1 = train[TARGETS + features].copy()

train_iteration_1.head(3)

,event,time_to_hit_hours,area_first_ha,area_growth_abs_0_5h,area_growth_rel_0_5h,centroid_displacement_m,centroid_speed_m_per_h,spread_bearing_sin,spread_bearing_cos,num_perimeters_0_5h,dt_first_last_0_5h,low_temporal_resolution_0_5h
218,0,44.011253,71.946930,0.00000,0.000000,0.000000,0.000000,0.000000,1.00000,2,3.710653,0
113,1,3.791231,39.334108,130.77579,3.324743,514.212375,114.291842,0.903783,0.42799,7,4.499117,0
140,0,34.533803,9.314299,0.00000,0.000000,0.000000,0.000000,0.000000,1.00000,2,3.768030,0


In [39]:
from lifelines import CoxPHFitter
# train the model on the training data
cph = CoxPHFitter()
cph.fit(train_iteration_1, duration_col='time_to_hit_hours', event_col='event')

<lifelines.CoxPHFitter: fitted with 176 total observations, 125 right-censored observations>

In [40]:
from lifelines.utils import concordance_index
from sksurv.metrics import brier_score
def evaluate_model(validation_set, model):
    """
    Evaluate the accuracy of the model. Uses generate_predictions() function to generate probability predictions
    :param validation_set: DataFrame whose probabilities must be estimated at the key time horizons
    :param model: The model to be evaluated
    :return: Hybrid Score in accordance with the datathon description on Kaggle: https://www.kaggle.com/competitions/WiDSWorldWide_GlobalDathon26/overview
    """
    # separate the predictors from the response variables
    X = validation_set[features]
    y = validation_set[TARGETS]

    TIME_COLUMN = 'time_to_hit_hours'
    EVENT_COLUMN = 'event'
    # 1️⃣ Predict survival function for each row using the trained model
    surv_funcs = model.predict_survival_function(X)

    predictions_at_time_horizons = generate_predictions(surv_funcs, surv_funcs.columns)

    # now actually evaluate the predictions
    # Calculate Concordance Index (higher is better, max 1.0)
    c_index = concordance_index(
        event_observed=y[EVENT_COLUMN],
        predicted_scores=model.predict_partial_hazard(X),
        event_times=y[TIME_COLUMN]
    )
    print(f"Concordance Index: {c_index:.4f}")

    # Calculate Brier Score for each time horizon (lower is better, min 0.0)
    brier_scores = {}
    for time in TIME_HORIZONS:
        col_name = f'prob_{time}h'
        # Create binary outcome: 1 if event occurred within time horizon, 0 otherwise
        y_binary = (y[EVENT_COLUMN] == 1) & (y[TIME_COLUMN] <= time)
        y_pred = predictions_at_time_horizons[col_name].values

        # Calculate Brier Score
        brier = np.mean((y_pred - y_binary.astype(int))**2)
        brier_scores[time] = brier
        print(f"Brier Score at {time}h: {brier:.4f}")

    # Calculate weighted Brier Score per competition specifications:
    # 24h: 30%, 48h: 40%, 72h: 30% (12h is excluded)
    weighted_brier = (0.3 * brier_scores[24] +
                      0.4 * brier_scores[48] +
                      0.3 * brier_scores[72])

    # Calculate hybrid score per competition requirements:
    # Hybrid Score = 0.3 * Concordance Index + 0.7 * (1 - Weighted Brier Score)
    hybrid_score = (0.3 * c_index) + (0.7 * (1 - weighted_brier))
    print(f"\nWeighted Brier Score: {weighted_brier:.4f}")
    print(f"Hybrid Score: {hybrid_score:.4f}")

    # Return predictions and metrics
    results = {
        'predictions': predictions_at_time_horizons,
        'concordance_index': c_index,
        'brier_scores': brier_scores,
        'hybrid_score': hybrid_score
    }

    return results

In [41]:
def generate_predictions(raw_probability_predictions, event_ids):
    """
    Generate probability predictions for each row
    :param raw_probability_predictions: DataFrame with survival probabilities (index=time, columns=event indices)
    :param event_ids: Event IDs corresponding to each column in raw_probability_predictions
    :return: DataFrame with probability predictions at each time horizon
    """
    submission_data = []
    # iterate through each row (event)
    for col in raw_probability_predictions.columns:
        # interpolate the survival probabilities at the specified time horizons
        survival_probs = np.interp(
            TIME_HORIZONS,
            raw_probability_predictions.index.values,
            raw_probability_predictions[col].values
        )
        # probability of being hit is the complement of the probability of surviving
        prob_hit = 1 - survival_probs
        submission_data.append(prob_hit)

    # wrap all results into a dataframe
    submission_df = pd.DataFrame(submission_data, columns=['prob_12h', 'prob_24h', 'prob_48h', 'prob_72h'])
    print('Created predictions DataFrame with dimensions:', submission_df.shape)
    # Add event_id (convert to list if needed)
    submission_df['event_id'] = list(event_ids)

    # Reorder columns to match submission format
    submission_df = submission_df[['event_id', 'prob_12h', 'prob_24h', 'prob_48h', 'prob_72h']]
    return submission_df

In [42]:
evaluate_model(validation_set=val, model=cph)

Created predictions DataFrame with dimensions: (45, 4)
Concordance Index: 0.2299
Brier Score at 12h: 0.1002
Brier Score at 24h: 0.1531
Brier Score at 48h: 0.1543
Brier Score at 72h: 0.1968

Weighted Brier Score: 0.1667
Hybrid Score: 0.6523


{'predictions':     event_id  prob_12h  prob_24h  prob_48h  prob_72h
 0        132  0.155320  0.213927  0.229885  0.531820
 1        148  0.265069  0.355434  0.379108  0.749588
 2         93  0.162486  0.223420  0.239971  0.549414
 3        180  0.999972  1.000000  1.000000  1.000000
 4         15  0.071070  0.099791  0.107818  0.282118
 5        115  0.302986  0.402329  0.427974  0.802655
 6        172  0.161064  0.221539  0.237974  0.545965
 7        209  0.108451  0.151001  0.162760  0.403161
 8         75  0.158595  0.218269  0.234500  0.539925
 9        142  0.153544  0.211570  0.227378  0.527378
 10       100  0.129798  0.179841  0.193579  0.464775
 11        30  0.407031  0.525383  0.554583  0.904596
 12       190  0.489479  0.616623  0.646695  0.951331
 13         9  0.162123  0.222940  0.239462  0.548536
 14        67  0.162284  0.223152  0.239687  0.548925
 15       219  0.161011  0.221468  0.237899  0.545835
 16       175  0.133726  0.185116  0.199206  0.475553
 17        18